# 15 · Discovering a battery capacity-fade law

Given battery cycle data, can we recover a compact, *interpretable* degradation
law instead of a black-box predictor? Using omnibias closed-form operator
channels we fit a sparse relation for the capacity derivative `dq/dn` and roll it
out into a monotone fade curve.

This notebook runs on the reproducible **synthetic-cycle** generator; the real
Severson (2019) path is in `examples/symbolic_discovery/battery_law_discovery/`
(`download_severson.py`).

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
sys.path.insert(0, "..")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

from examples.symbolic_discovery.battery_law_discovery.severson_loader import make_synthetic_cycle_table
from examples.symbolic_discovery.battery_law_discovery.omnibias_law_model import (
    build_operator_library, fit_sparse_law, predict_law,
    build_physics_library, fit_physics_constrained_law, rollout_law,
)

## Synthetic cycling data

A handful of cells, each with a normalized capacity that fades with cycle count.

In [ ]:
table = make_synthetic_cycle_table(n_cells=5, n_cycles=80, seed=3)
fig, ax = plt.subplots(figsize=(7.2, 4.0))
cell_ids = np.asarray(table.require("cell_id"))
for cid in sorted(set(cell_ids.tolist())):
    mask = cell_ids == cid
    n = np.asarray(table.require("cycle_norm"), float)[mask]
    q = np.asarray(table.require("capacity_norm"), float)[mask]
    ax.plot(n, q, alpha=0.8)
ax.set_xlabel("normalized cycle"); ax.set_ylabel("normalized capacity")
ax.set_title("Synthetic capacity fade across cells")
plt.tight_layout()

## Recover the degradation law

On a clean exponential-fade reference, the sparse operator fit recovers a
first-order decay `dq/dn ∝ q` with the correct (negative) sign to high accuracy.

In [ ]:
n = np.linspace(0.0, 1.0, 200)
k = 0.17
q = np.exp(-k * n)
dqdn = -k * q
d2qdn2 = k * k * q
x = np.zeros((n.size, 2)); x[:, 0] = n
library, names = build_operator_library(x, q=q, d2qdn2=d2qdn2)
law = fit_sparse_law(library, dqdn, names, threshold=1e-4)
pred = predict_law(law, library)
rmse = float(np.sqrt(np.mean((pred - dqdn) ** 2)))
print(f"dq/dn fit RMSE = {rmse:.2e};  coef[q] = {law.coef[names.index('q')]:.4f}  (sign < 0 ✓)")

phys_lib, phys_names = build_physics_library(x, q=q)
phys_law = fit_physics_constrained_law(phys_lib, dqdn, q, phys_names, threshold=1e-6)
rolled = rollout_law(phys_law, x[0], float(q[0]), n)

fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.0))
axl.plot(n, dqdn, color=PRIMARY, label="true dq/dn")
axl.plot(n, pred, color=ACCENT, ls="--", label="recovered")
axl.set_title("Recovered capacity-derivative law"); axl.set_xlabel("cycle"); axl.legend()
axr.plot(n, q, color=PRIMARY, label="true capacity")
axr.plot(n, rolled, color=GOOD, ls="--", label="physics-law rollout")
axr.set_title("Monotone fade rollout"); axr.set_xlabel("cycle"); axr.legend()
plt.tight_layout()
assert np.all(np.diff(rolled) <= 1e-12)  # monotone non-increasing

## Takeaway

The discovered law is a one-line, physically meaningful relation — not a black
box — and its rollout reproduces the monotone capacity fade. The same operator
library runs on real Severson cells via the experiment's loader.